# Diabetes Preprocessing
---

## Load Data

In [22]:
import pandas as pd

RAW = "../data/raw"
df = pd.read_csv(f"{RAW}/diabetic_data.csv")

C:\Users\cameron.letendre\AppData\Local\Temp\ipykernel_42552\2929803960.py:4: DtypeWarning: Columns (10: payer_code) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(f"{RAW}/diabetic_data.csv")


### Fill in blanks with a name

most "missing" values are not really missing. 
- A blank A1C result means the test was never ordered. 
- A blank payer or specialty means it was not recorded. 

Those are categories, so we give them names right away instead of leaving them as NaN where later steps would silently drop or impute them.

In [23]:
pre = df.isna().sum()

# Blank lab results mean the test was not ordered
df[["A1Cresult", "max_glu_serum"]] = df[["A1Cresult", "max_glu_serum"]].fillna("NotTested")

# Blank text fields mean "not recorded", which is still a category
for col in ["race", "payer_code", "medical_specialty", "diag_1", "diag_2", "diag_3"]:
    df[col] = df[col].fillna("Missing")

filled = df.isna().sum()
changed = pre[pre != filled]
print(changed)


race                  2273
payer_code           40256
medical_specialty    49949
diag_1                  21
diag_2                 358
diag_3                1423
max_glu_serum        96420
A1Cresult            84748
dtype: int64


### Remove encounters that cannot be readmitted
A patient who died in hospital or went to hospice cannot come back within 30 days.

In [24]:
print(f"{len(df):,} encounters, {df['patient_nbr'].nunique():,} patients")
not_readmittable = [11, 13, 14, 19, 20, 21]

df = df[~df["discharge_disposition_id"].isin(not_readmittable)]
df = df.reset_index(drop=True)

print(f"{len(df):,} encounters, {df['patient_nbr'].nunique():,} patients")

101,766 encounters, 71,518 patients
99,343 encounters, 69,990 patients


removing small instance of encounters whose gender is recorded as "Unknown/Invalid"
miniscule and data entry error

In [25]:
print(f"{len(df):,} encounters, {df['patient_nbr'].nunique():,} patients")
df = df[df["gender"] != "Unknown/Invalid"]
df = df.reset_index(drop=True)
print(f"{len(df):,} encounters, {df['patient_nbr'].nunique():,} patients")

99,343 encounters, 69,990 patients
99,340 encounters, 69,987 patients


### Build Target
target is binary, readmitted in under 30 days versus everything else. anything after 30 days is a negative creating a 